In [23]:
import time

import pandas as pd
from sklearn.metrics import roc_auc_score

train = pd.read_csv("../train.csv")
validation = pd.read_csv("../validation.csv")
test = pd.read_csv("../test.csv")

feature_columns = [c for c in train.columns if c not in ["row_id", "target"]]

print("Размер train выборки:", train.shape)
print("Размер validation выборки:", validation.shape)
print("Размер test выборки:", test.shape)
print("Количество features:", len(feature_columns))

Размер train выборки: (2520, 514)
Размер validation выборки: (560, 514)
Размер test выборки: (1116, 513)
Количество features: 512


In [24]:
print("Баланс классов:")
print(train["target"].value_counts())

print("\nДоли классов:")
print(train["target"].value_counts(normalize=True))

print("\nКоличество пропусков:", train.isna().sum().sum())
print("Количество дубликатов:", train.duplicated().sum())

constant_features = [
    c for c in feature_columns
    if train[c].nunique() <= 1
]

print("Количество const признаков:", len(constant_features))

Баланс классов:
target
1    1260
0    1260
Name: count, dtype: int64

Доли классов:
target
1    0.5
0    0.5
Name: proportion, dtype: float64

Количество пропусков: 0
Количество дубликатов: 0
Количество const признаков: 0


In [25]:
def evaluate_model(name, model):
    start = time.perf_counter()

    model.fit(train[feature_columns], train["target"])

    probability = model.predict_proba(validation[feature_columns])[:, 1]

    auc = roc_auc_score(validation["target"], probability)

    elapsed = time.perf_counter() - start

    return {
        "model": name,
        "roc_auc": auc,
        "time_sec": elapsed,
    }

In [26]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 20260916

models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=3,
        min_samples_leaf=20,
        random_state=SEED,
    ),
    "Logistic Regression": Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    max_iter=2000,
                    random_state=SEED,
                ),
            ),
        ]
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=SEED,
        n_jobs=-1,
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=SEED,
    ),
}

In [32]:
results = []

for name, model in models.items():
    result = evaluate_model(name, model)
    results.append(result)

results_df = (
    pd.DataFrame(results)
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)

results_df

,model,roc_auc,time_sec
0,Gradient Boosting,0.905051,15.691896
1,Random Forest,0.904534,0.357717
2,Logistic Regression,0.883916,0.021268
3,Decision Tree,0.747309,0.181323


## Базовое сравнение

Сравнил 4 модели на одних данных

Все модели обучались на `train` 

Качество проверялось на `validation`

Сравниваю ROC-AUC и время работы

## Вывод по базовым моделям

У Decision Tree: 0.747

У Logistic Regression: 0.884

Лучшие рез-ты у ансамблей:
Random Forest: 0.905, Gradient Boosting: 0.905

Gradient Boosting чуть лучше, зато Random Forest работал быстрее

=> дальше я решил подробнее вникнуть в Random Forest

In [28]:
from sklearn.ensemble import RandomForestClassifier

rf_results = []

for max_depth in [3, 5, 10, 20, None]:
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=max_depth,
        random_state=SEED,
        n_jobs=-1,
    )

    result = evaluate_model(
        f"RF max_depth={max_depth}",
        model,
    )

    result["max_depth"] = max_depth
    rf_results.append(result)

rf_depth_df = pd.DataFrame(rf_results)
rf_depth_df

,model,roc_auc,time_sec,max_depth
0,RF max_depth=3,0.876186,0.200731,3.0
1,RF max_depth=5,0.881276,0.212413,5.0
2,RF max_depth=10,0.900089,0.321265,10.0
3,RF max_depth=20,0.904362,0.308592,20.0
4,RF max_depth=None,0.904534,0.311733,NaN


### Вывод по max_depth

При увеличении глубины ROC-AUC рос

После примерно 20 улучшение стало маленьким

Лучший рез-т - без ограничения глубины, => дальше `max_depth=None`.

In [29]:
rf_feature_results = []

for max_features in ["sqrt", "log2", 0.25, 0.5, 1.0]:
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        max_features=max_features,
        random_state=SEED,
        n_jobs=-1,
    )

    result = evaluate_model(
        f"RF max_features={max_features}",
        model,
    )

    result["max_features"] = max_features
    rf_feature_results.append(result)

rf_features_df = pd.DataFrame(rf_feature_results)
rf_features_df

,model,roc_auc,time_sec,max_features
0,RF max_features=sqrt,0.904534,0.318607,sqrt
1,RF max_features=log2,0.892423,0.186665,log2
2,RF max_features=0.25,0.901052,1.575281,0.25
3,RF max_features=0.5,0.906696,3.238596,0.5
4,RF max_features=1.0,0.907321,6.607959,1.0


### Вывод по max_features

При увеличении `max_features` качество в целом улучшалось

Лучший ROC-AUC - при `max_features=1.0`

`0.5` дало похожий рез-т, но быстрее

Далее `max_features=1.0`

In [30]:
rf_leaf_results = []

for min_samples_leaf in [1, 2, 5, 10, 20]:
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        max_features=1.0,
        min_samples_leaf=min_samples_leaf,
        random_state=SEED,
        n_jobs=-1,
    )

    result = evaluate_model(
        f"RF min_samples_leaf={min_samples_leaf}",
        model,
    )

    result["min_samples_leaf"] = min_samples_leaf
    rf_leaf_results.append(result)

rf_leaf_df = pd.DataFrame(rf_leaf_results)
rf_leaf_df

,model,roc_auc,time_sec,min_samples_leaf
0,RF min_samples_leaf=1,0.907321,6.407671,1
1,RF min_samples_leaf=2,0.908457,6.382068,2
2,RF min_samples_leaf=5,0.905765,5.159234,5
3,RF min_samples_leaf=10,0.903495,4.643984,10
4,RF min_samples_leaf=20,0.900281,3.946733,20


### Вывод по min_samples_leaf

Лучший - при `min_samples_leaf=2`

При увеличении ROC-AUC начал падать

Дальше `min_samples_leaf=2`

In [31]:
rf_estimators_results = []

for n_estimators in [50, 100, 200, 500, 1000]:
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=None,
        max_features=1.0,
        min_samples_leaf=2,
        random_state=SEED,
        n_jobs=-1,
    )

    result = evaluate_model(
        f"RF n_estimators={n_estimators}",
        model,
    )

    result["n_estimators"] = n_estimators
    rf_estimators_results.append(result)

rf_estimators_df = pd.DataFrame(rf_estimators_results)
rf_estimators_df

,model,roc_auc,time_sec,n_estimators
0,RF n_estimators=50,0.899018,1.583026,50
1,RF n_estimators=100,0.904171,3.070025,100
2,RF n_estimators=200,0.908457,5.985254,200
3,RF n_estimators=500,0.913431,15.188921,500
4,RF n_estimators=1000,0.911837,29.702556,1000


### Вывод по n_estimators

При увеличении деревьев качество сначала росло

Лучший - при `n_estimators=500`: 0.9134

При 1000 деревьях качество упало при времени 2 раза больше

Дальше `n_estimators=500`

In [34]:
rf_criterion_results = []

for criterion in ["gini", "entropy", "log_loss"]:
    model = RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        max_features=1.0,
        min_samples_leaf=2,
        criterion=criterion,
        random_state=SEED,
        n_jobs=-1,
    )

    result = evaluate_model(
        f"RF criterion={criterion}",
        model,
    )

    result["criterion"] = criterion
    rf_criterion_results.append(result)

rf_criterion_df = pd.DataFrame(rf_criterion_results)
rf_criterion_df

,model,roc_auc,time_sec,criterion
0,RF criterion=gini,0.913425,14.416020,gini
1,RF criterion=entropy,0.909949,13.611486,entropy
2,RF criterion=log_loss,0.909943,14.313310,log_loss


### Вывод по criterion

Лучший - с `criterion="gini"`: 0.9134

`entropy` и `log_loss` -  дали рез-т похуже

Дальше `criterion="gini"`

In [35]:
final_rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    max_features=1.0,
    min_samples_leaf=2,
    criterion="gini",
    random_state=SEED,
    n_jobs=-1,
)

final_rf_result = evaluate_model(
    "Final Random Forest",
    final_rf,
)

pd.DataFrame([final_rf_result])

,model,roc_auc,time_sec
0,Final Random Forest,0.913425,13.953143


## Итог по Random Forest

Базовый Random Forest был с ROC-AUC - 0.9045

После подбора гиперпараметров лучший результат вырос до 0.9134

Лучшие параметры:

- `n_estimators=500`
- `max_depth=None`
- `max_features=1.0`
- `min_samples_leaf=2`
- `criterion="gini"`

На этом подбор параметров Random Forest останавливаю